In [1]:
!pip install pymysql

In [2]:
import pymysql
import pandas as pd

In [3]:
db_config = {
    "host": "localhost", 
    "user": "root",      
    "password": "2007", 
    "database": "sales"  
}

In [4]:
queries ={
    " TASK 1- Finding the total revenue of year 2022*" :
"""SELECT 
    DATE_FORMAT(order_date, '%m') AS month_number,
    DATE_FORMAT(order_date, '%M') AS month_name,
    SUM(before_discount) AS total_revenue
FROM 
    order_detail
WHERE 
    order_date >= '2022-01-01' AND order_date < '2023-01-01'
GROUP BY 
    month_number,
    month_name
ORDER BY 
    month_number;""",
 " TASK 2- Finding which payment method contributed the most among all " :
 """SELECT 
    pd.payment_method,
    SUM(od.before_discount) AS total_sales,
    SUM(od.qty_ordered) AS total_quantity
FROM 
    order_detail od
JOIN 
    payment_detail pd ON od.payment_id = pd.id
WHERE 
    od.is_valid = 1
GROUP BY 
    pd.payment_method
ORDER BY 
    total_sales DESC;""",
    
    "TASK 3-Finding the average order value of every mouth of 2022" :
   """ SELECT 
    DATE_FORMAT(order_date, '%m') AS month_number,
    DATE_FORMAT(order_date, '%M') AS month_name,
    SUM(before_discount) / COUNT(id) AS average_order_value
FROM 
    order_detail
WHERE 
    is_valid = 1 
    AND order_date >= '2022-01-01' AND order_date < '2023-01-01'
GROUP BY 
    month_number,
    month_name
ORDER BY 
    month_number;""",
    
    "TASK 4-Finding the Net revenue,Total revenue and discount of every category " :
   """ SELECT 
    sd.category,
    SUM(od.before_discount) AS total_revenue,
    SUM(od.after_discount) AS net_revenue,
    SUM(od.before_discount - od.after_discount) AS discount_impact
FROM 
    order_detail od
JOIN 
    sku_detail sd ON od.sku_id = sd.id
WHERE 
    od.is_valid = 1
GROUP BY 
    sd.category
ORDER BY 
    discount_impact DESC;""",
    
    "TASK 5-Finding the product with the largest sales drop from 2021 to 2022" :
    
   """ WITH YearlySales AS (
    SELECT 
        sd.sku_name,
        SUM(CASE WHEN od.order_date >= '2021-01-01' AND od.order_date < '2022-01-01' THEN od.after_discount ELSE 0 END) AS sales_2021,
        SUM(CASE WHEN od.order_date >= '2022-01-01' AND od.order_date < '2023-01-01' THEN od.after_discount ELSE 0 END) AS sales_2022
    FROM 
        order_detail od
    JOIN 
        sku_detail sd ON od.sku_id = sd.id
    WHERE 
        od.is_valid = 1
        AND od.order_date >= '2021-01-01' AND od.order_date < '2023-01-01'
    GROUP BY 
        sd.sku_name
)
SELECT 
    sku_name,
    sales_2021,
    sales_2022,
    (sales_2021 - sales_2022) AS sales_decrease
FROM 
    YearlySales
WHERE 
    (sales_2021 - sales_2022) > 0
ORDER BY 
    sales_decrease DESC
LIMIT 10;""",

"TASK 6-Comparing the sales of weekday and weekends of Q4 to find which one is productive" :

"""SELECT 
    DATE_FORMAT(order_date, '%M') AS time_period,
    SUM(CASE WHEN DAYOFWEEK(order_date) IN (1, 7) THEN before_discount ELSE 0 END) / 
        NULLIF(COUNT(DISTINCT CASE WHEN DAYOFWEEK(order_date) IN (1, 7) THEN order_date END), 0) AS avg_weekend_sales,
    SUM(CASE WHEN DAYOFWEEK(order_date) BETWEEN 2 AND 6 THEN before_discount ELSE 0 END) / 
        NULLIF(COUNT(DISTINCT CASE WHEN DAYOFWEEK(order_date) BETWEEN 2 AND 6 THEN order_date END), 0) AS avg_weekday_sales
FROM 
    order_detail
WHERE 
    is_valid = 1 
    AND order_date >= '2022-10-01' AND order_date < '2023-01-01'
GROUP BY 
    DATE_FORMAT(order_date, '%m'), 
    DATE_FORMAT(order_date, '%M')

UNION ALL
SELECT 
    'Q4 Overall' AS time_period,
    SUM(CASE WHEN DAYOFWEEK(order_date) IN (1, 7) THEN before_discount ELSE 0 END) / 
        NULLIF(COUNT(DISTINCT CASE WHEN DAYOFWEEK(order_date) IN (1, 7) THEN order_date END), 0) AS avg_weekend_sales,
    SUM(CASE WHEN DAYOFWEEK(order_date) BETWEEN 2 AND 6 THEN before_discount ELSE 0 END) / 
        NULLIF(COUNT(DISTINCT CASE WHEN DAYOFWEEK(order_date) BETWEEN 2 AND 6 THEN order_date END), 0) AS avg_weekday_sales
FROM 
    order_detail
WHERE 
    is_valid = 1 
    AND order_date >= '2022-10-01' AND order_date < '2023-01-01';""",
}

In [5]:
from IPython.display import display

In [6]:
try:
    connection = pymysql.connect(**db_config)
    print("Database connection successful!")
    
    # Execute each query and display results
    for question, query in queries.items():
        print(f"\n--- {question} ---")
        df = pd.read_sql(query, connection)
        display(df)  # Display the DataFrame in the notebook
        
except Exception as e:
    print(f"An error occurred: {e}")
finally:
    if connection:
        connection.close()
        print("Database connection closed.")

Database connection successful!

---  TASK 1- Finding the total revenue of year 2022* ---


C:\Users\DELL\AppData\Local\Temp\ipykernel_10128\4170545264.py:8: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df = pd.read_sql(query, connection)


,month_number,month_name,total_revenue
0,01,January,4.348054e+08
1,02,February,3.608317e+08
2,03,March,4.996432e+08
3,04,April,6.797711e+08
4,05,May,5.333523e+08
5,06,June,3.919410e+08
6,07,July,4.439048e+08
7,08,August,6.864309e+08
8,09,September,6.583834e+08
9,10,October,1.243432e+08



---  TASK 2- Finding which payment method contributed the most among all  ---


C:\Users\DELL\AppData\Local\Temp\ipykernel_10128\4170545264.py:8: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df = pd.read_sql(query, connection)


...

,sku_name,sales_2021,sales_2022,sales_decrease
0,iPhone SE-16GB,50252128.0,0.0,50252128.0
1,samsungGALAXY S-7 EDGE 32GB LTE,41739294.0,0.0,41739294.0
2,Apple iPhone 6 (128GB) Gold,33581652.0,0.0,33581652.0
3,Apple iPhone 6 Plus (64GB) Gold,27984710.0,0.0,27984710.0
4,Apple iPhone 6S Plus 16GB Silver,39601240.0,16791000.0,22810240.0
5,Apple iPhone 6S Plus 16GB,21865768.0,0.0,21865768.0
6,samsungGALAXY S-7 32GB LTE,20415826.0,0.0,20415826.0
7,Huawei P8 lite,18834166.0,0.0,18834166.0
8,samsungGALAXY NOTE-5 LTE 32GB,16704000.0,0.0,16704000.0
9,Samsung Galaxy S6 Edge Plus,15659826.0,0.0,15659826.0



--- TASK 6-Comparing the sales of weekday and weekends of Q4 to find which one is productive ---


C:\Users\DELL\AppData\Local\Temp\ipykernel_10128\4170545264.py:8: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df = pd.read_sql(query, connection)


,time_period,avg_weekend_sales,avg_weekday_sales
0,October,5.708341e+06,7.793912e+06
1,November,5.774045e+06,6.204666e+06
2,December,4.105994e+06,8.411063e+06
3,Q4 Overall,5.269300e+06,7.450820e+06


Database connection closed.


In [7]:
import os

In [8]:
output_folder = r"C:\Users\DELL\anaconda_projects\db\Google Looker Studio\dataset"  
os.makedirs(output_folder, exist_ok=True)  

In [9]:
tables = ["order_detail", "sku_detail", "payment_detail","customer_detail"] 

In [10]:
try:
    # Connect to the database
    connection = pymysql.connect(**db_config)
    print("Database connection successful!")

    for table in tables:
        print(f"Exporting table: {table}")
        
        # SQL query to fetch all data from the table
        query = f"SELECT * FROM {table};"
        
        # Read table data into a DataFrame
        df = pd.read_sql(query, connection)
        
        # Save the DataFrame to a CSV file
        output_file = os.path.join(output_folder, f"{table}.csv")
        df.to_csv(output_file, index=False)
        
        print(f"Table {table} exported successfully to {output_file}.")

except Exception as e:
    print(f"An error occurred: {e}")
finally:
    if connection:
        connection.close()
        print("Database connection closed.")

Database connection successful!
Exporting table: order_detail
Table order_detail exported successfully to C:\Users\DELL\anaconda_projects\db\Google Looker Studio\dataset\order_detail.csv.
Exporting table: sku_detail
Table sku_detail exported successfully to C:\Users\DELL\anaconda_projects\db\Google Looker Studio\dataset\sku_detail.csv.
Exporting table: payment_detail
Table payment_detail exported successfully to C:\Users\DELL\anaconda_projects\db\Google Looker Studio\dataset\payment_detail.csv.
Exporting table: customer_detail
Table customer_detail exported successfully to C:\Users\DELL\anaconda_projects\db\Google Looker Studio\dataset\customer_detail.csv.
Database connection closed.


C:\Users\DELL\AppData\Local\Temp\ipykernel_10128\1590211234.py:13: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df = pd.read_sql(query, connection)


In [11]:
df_od = pd.read_csv("order_detail.csv")
df_sd = pd.read_csv("sku_detail.csv")
df_pd = pd.read_csv("payment_detail.csv")
df_cd = pd.read_csv("customer_detail.csv")

In [12]:
df_sd.rename(columns={'id':'sku_id'}, inplace=True)
df_cd.rename(columns={'id':'customer_id'}, inplace=True)
df_pd.rename(columns={'id':'payment_id'}, inplace=True)

In [13]:
df = pd.DataFrame(df_od\
                  .merge(df_sd, how='left', on='sku_id')\
                  .merge(df_cd, how='left', on='customer_id')\
                  .merge(df_pd, how='left', on='payment_id')
                  )

In [14]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 5764 entries, 0 to 5763
Data columns (total 19 columns):
 #   Column           Non-Null Count  Dtype  
---  ------           --------------  -----  
 0   id               5764 non-null   object 
 1   customer_id      5764 non-null   object 
 2   order_date       5764 non-null   object 
 3   sku_id           5764 non-null   object 
 4   price            5764 non-null   int64  
 5   qty_ordered      5764 non-null   int64  
 6   before_discount  5764 non-null   float64
 7   discount_amount  5764 non-null   float64
 8   after_discount   5764 non-null   float64
 9   is_gross         5764 non-null   int64  
 10  is_valid         5764 non-null   int64  
 11  is_net           5764 non-null   int64  
 12  payment_id       5764 non-null   int64  
 13  sku_name         5764 non-null   object 
 14  base_price       5764 non-null   float64
 15  cogs             5764 non-null   float64
 16  category         5764 non-null   object 
 17  registered_dat

In [15]:
print(df_pd.columns)

Index(['payment_id', 'payment_method'], dtype='object')


In [16]:
print(df_od.columns)

Index(['id', 'customer_id', 'order_date', 'sku_id', 'price', 'qty_ordered',
       'before_discount', 'discount_amount', 'after_discount', 'is_gross',
       'is_valid', 'is_net', 'payment_id'],
      dtype='object')


In [17]:
df_pd['payment_id'] = df_pd['payment_id'].astype(str) 
df_od['id'] = df_od['id'].astype(str)                  
df_sample = pd.merge(df_pd, df_od, how='left', left_on='payment_id', right_on='id')
df_sample.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 16 entries, 0 to 15
Data columns (total 15 columns):
 #   Column           Non-Null Count  Dtype  
---  ------           --------------  -----  
 0   payment_id_x     16 non-null     object 
 1   payment_method   16 non-null     object 
 2   id               0 non-null      object 
 3   customer_id      0 non-null      object 
 4   order_date       0 non-null      object 
 5   sku_id           0 non-null      object 
 6   price            0 non-null      float64
 7   qty_ordered      0 non-null      float64
 8   before_discount  0 non-null      float64
 9   discount_amount  0 non-null      float64
 10  after_discount   0 non-null      float64
 11  is_gross         0 non-null      float64
 12  is_valid         0 non-null      float64
 13  is_net           0 non-null      float64
 14  payment_id_y     0 non-null      float64
dtypes: float64(9), object(6)
memory usage: 2.0+ KB


In [18]:
for x in ['order_date', 'registered_date']:
  df[x] = pd.to_datetime(df[x])

df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 5764 entries, 0 to 5763
Data columns (total 19 columns):
 #   Column           Non-Null Count  Dtype         
---  ------           --------------  -----         
 0   id               5764 non-null   object        
 1   customer_id      5764 non-null   object        
 2   order_date       5764 non-null   datetime64[ns]
 3   sku_id           5764 non-null   object        
 4   price            5764 non-null   int64         
 5   qty_ordered      5764 non-null   int64         
 6   before_discount  5764 non-null   float64       
 7   discount_amount  5764 non-null   float64       
 8   after_discount   5764 non-null   float64       
 9   is_gross         5764 non-null   int64         
 10  is_valid         5764 non-null   int64         
 11  is_net           5764 non-null   int64         
 12  payment_id       5764 non-null   int64         
 13  sku_name         5764 non-null   object        
 14  base_price       5764 non-null   float64

In [22]:
import os

file_path = r"C:\Users\DELL\Downloads\Google Looker project\dataset\finaldataset(1).csv" 

# Extract the directory path from the full file path
directory = os.path.dirname(file_path)

# Create the directory if it doesn't exist
os.makedirs(directory, exist_ok=True)

# Now save the file - the directory is guaranteed to exist
df.to_csv(file_path, index=False)

print(f"File successfully saved to {file_path}")

File successfully saved to C:\Users\DELL\Downloads\Google Looker project\dataset\finaldataset(1).csv
